# M1 · 01 · Environment Check
**Run this once after `bash setup_m1.sh` to confirm everything is correctly installed.**

Expected outcome: every cell runs green. No GPU is needed — M1 runs the GNN on CPU (PyG scatter ops have partial MPS support only). MPS availability is logged but not required.

---

## 1 · Python & Conda

In [2]:
import sys, platform, subprocess
print(f"Python   : {sys.version}")
print(f"Platform : {platform.platform()}")
print(f"Machine  : {platform.machine()}")

# Confirm we are inside the compiler_opt conda env
env = subprocess.run(['conda', 'info', '--envs'], capture_output=True, text=True)
active = [l for l in env.stdout.splitlines() if '*' in l]
print(f"Active env: {active[0] if active else 'UNKNOWN — activate compiler_opt first!'}")

Python   : 3.10.20 (main, Mar 11 2026, 17:43:48) [Clang 20.1.8 ]
Platform : macOS-26.3-arm64-arm-64bit
Machine  : arm64
Active env: # * -> active


## 2 · PyTorch + MPS

In [3]:
import torch
print(f"PyTorch version : {torch.__version__}")
print(f"MPS available   : {torch.backends.mps.is_available()}")
print(f"MPS built       : {torch.backends.mps.is_built()}")

# Quick MPS tensor test (for info only — we will NOT use MPS for GNN)
if torch.backends.mps.is_available():
    x = torch.ones(3, 3, device='mps')
    print(f"MPS tensor ok   : {x.sum().item() == 9.0}")
else:
    print("MPS not available — CPU training will be used (expected on some M1 configs)")

PyTorch version : 2.5.1
MPS available   : True
MPS built       : True
MPS tensor ok   : True


## 3 · PyTorch Geometric

In [4]:
import torch_geometric as pyg
from torch_geometric.data import HeteroData
from torch_geometric.nn import HeteroConv, GATConv
print(f"PyTorch Geometric : {pyg.__version__}")

# Build a minimal HeteroData and run a forward pass to confirm scatter ops work
data = HeteroData()
data['a'].x = torch.randn(4, 8)
data['b'].x = torch.randn(3, 8)
data['a', 'to', 'b'].edge_index = torch.tensor([[0,1,2],[0,1,2]])

conv = HeteroConv({('a','to','b'): GATConv((-1,-1), 16, heads=2, concat=False, add_self_loops=False)})
out = conv(data.x_dict, data.edge_index_dict)
print(f"HeteroConv output shape : {out['b'].shape}  (expected torch.Size([3, 16]))")
assert out['b'].shape == torch.Size([3, 16]), "Shape mismatch!"
print("PyTorch Geometric : OK")

/opt/anaconda3/envs/compiler_opt/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PyTorch Geometric : 2.7.0
HeteroConv output shape : torch.Size([3, 16])  (expected torch.Size([3, 16]))
PyTorch Geometric : OK


/var/folders/jf/bzq8rwjs1cd5r5v9j9zy87_00000gn/T/ipykernel_1979/2725963134.py:12: UserWarning: There exist node types ({'a'}) whose representations do not get updated during message passing as they do not occur as destination type in any edge type. This may lead to unexpected behavior.
  conv = HeteroConv({('a','to','b'): GATConv((-1,-1), 16, heads=2, concat=False, add_self_loops=False)})


## 4 · LLVM + Clang

In [6]:
import subprocess, shutil

def check_tool(name):
    path = shutil.which(name)
    if path is None:
        print(f"  {name:10s} : NOT FOUND — add LLVM to PATH (see setup_m1.sh)")
        return False
    r = subprocess.run([name, '--version'], capture_output=True, text=True)
    ver = r.stdout.splitlines()[0] if r.stdout else r.stderr.splitlines()[0]
    print(f"  {name:10s} : {path}")
    print(f"  {'':10s}   {ver}")
    return True

print("LLVM tools:")
clang_ok = check_tool('clang')
opt_ok   = check_tool('opt')
llc_ok   = check_tool('llc')

if not (clang_ok and opt_ok):
    print("\nFIX: Run the following and restart this notebook:")
    print('  echo \'export PATH="$(brew --prefix llvm@14)/bin:$PATH"\' >> ~/.zshrc && source ~/.zshrc')

LLVM tools:
  clang      : /opt/homebrew/opt/llvm@14/bin/clang
               Homebrew clang version 14.0.6
  opt        : /opt/homebrew/opt/llvm@14/bin/opt
               Homebrew LLVM version 14.0.6
  llc        : /opt/homebrew/opt/llvm@14/bin/llc
               Homebrew LLVM version 14.0.6


In [9]:
# Compile a tiny C program to IR and back — full pipeline smoke test
import tempfile, os

C_SRC = """
#include <stdio.h>
int add(int a, int b) { return a + b; }
int main() { printf("%d\\n", add(2, 3)); return 0; }
"""

with tempfile.TemporaryDirectory() as td:
    src = os.path.join(td, 'test.c')
    ll  = os.path.join(td, 'test.ll')
    opt_ll = os.path.join(td, 'test_opt.ll')
    
    with open(src, 'w') as f:
        f.write(C_SRC)

    # Compile to IR
    r1 = subprocess.run(
        ['clang', '-S', '-emit-llvm', '-O0',
         '-Xclang', '-disable-O0-optnone',
         '-fno-discard-value-names', src, '-o', ll],
        capture_output=True, text=True
    )
    if r1.returncode != 0:
        print("Initial clang invocation failed, attempting with SDK path...")
        # Dynamically get the SDK path using xcrun for macOS header inclusion
        sdk_result = subprocess.run(['xcrun', '--show-sdk-path'], capture_output=True, text=True)
        if sdk_result.returncode == 0:
            sdk_path = sdk_result.stdout.strip()
            r1 = subprocess.run(
                ['clang', '-S', '-emit-llvm', '-O0',
                 '-Xclang', '-disable-O0-optnone',
                 '-fno-discard-value-names', '-isysroot', sdk_path, src, '-o', ll],
                capture_output=True, text=True
            )
        else:
            print("Warning: xcrun failed to get SDK path, falling back to hardcoded sysroot. This may cause issues if the SDK is not at the expected location.")
            # Fallback: retry with hardcoded sysroot if xcrun fails
            r1 = subprocess.run(
                ['clang', '-S', '-emit-llvm', '-O0',
                 '-Xclang', '-disable-O0-optnone',
                 '-fno-discard-value-names', '-isysroot', '/Library/Developer/CommandLineTools/SDKs/MacOSX.sdk', src, '-o', ll],
                capture_output=True, text=True
            )
    assert r1.returncode == 0, f"clang failed: {r1.stderr}"
    print("  clang → .ll      : OK")

    # Apply mem2reg with opt
    r2 = subprocess.run(
        ['opt', '-mem2reg', '-S', ll, '-o', opt_ll],
        capture_output=True, text=True
    )
    assert r2.returncode == 0, f"opt failed: {r2.stderr}"
    print("  opt -mem2reg     : OK")

    # Count instructions
    with open(ll) as f:
        n_base = sum(1 for l in f if l.strip() and not l.strip().startswith(';'))
    with open(opt_ll) as f:
        n_opt = sum(1 for l in f if l.strip() and not l.strip().startswith(';'))
    print(f"  IR lines before  : {n_base}")
    print(f"  IR lines after   : {n_opt}")
    print(f"  Reduction        : {(1 - n_opt/n_base)*100:.1f}%")

print("\nLLVM pipeline : OK")

Initial clang invocation failed, attempting with SDK path...
  clang → .ll      : OK
  opt -mem2reg     : OK
  IR lines before  : 38
  IR lines after   : 30
  Reduction        : 21.1%

LLVM pipeline : OK


## 5 · Tree-sitter

In [10]:
from tree_sitter_languages import get_parser
parser = get_parser('c')
tree   = parser.parse(b'int f(int x) { return x + 1; }')
root   = tree.root_node
print(f"tree-sitter C parser : OK")
print(f"Root node type       : {root.type}")
print(f"Children             : {[c.type for c in root.children]}")

tree-sitter C parser : OK
Root node type       : translation_unit
Children             : ['function_definition']


## 6 · Other Dependencies

In [11]:
import importlib
REQUIRED = [
    ('networkx',     'networkx'),
    ('sklearn',      'scikit-learn'),
    ('matplotlib',   'matplotlib'),
    ('seaborn',      'seaborn'),
    ('pandas',       'pandas'),
    ('numpy',        'numpy'),
    ('tqdm',         'tqdm'),
    ('yaml',         'pyyaml'),
]
all_ok = True
for mod, pkg in REQUIRED:
    try:
        m = importlib.import_module(mod)
        ver = getattr(m, '__version__', '?')
        print(f"  {pkg:25s} {ver}")
    except ImportError:
        print(f"  {pkg:25s} MISSING — pip install {pkg}")
        all_ok = False

print()
print('All dependencies OK ✓' if all_ok else 'Some packages missing — re-run setup_m1.sh')

  networkx                  3.4.2
  scikit-learn              1.7.2
  matplotlib                3.10.9
  seaborn                   0.13.2
  pandas                    2.3.3
  numpy                     2.2.5
  tqdm                      4.67.3
  pyyaml                    6.0.3

All dependencies OK ✓


## 7 · Summary

| Component | Required | Notes |
|-----------|----------|-------|
| Python 3.10 | ✓ | conda env |
| PyTorch | ✓ | CPU used for GNN |
| MPS | optional | Dense layers only |
| PyTorch Geometric | ✓ | CPU wheels |
| clang / opt | ✓ | LLVM 14 via brew |
| tree-sitter | ✓ | C/C++ parser |

**If all checks passed:** open `M1_02_graph_builder.ipynb` next.

**If LLVM is missing:** `brew install llvm@14` then add it to PATH in `~/.zshrc`.